### Pipeline

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../train.csv")
df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [60]:
df = df.drop(columns=["PassengerId", "Name", "SibSp", "Parch", "Ticket", "Fare", "Cabin"])
df

,Survived,Pclass,Sex,Age,Embarked
0,0,3,male,22.0,S
1,1,1,female,38.0,C
2,1,3,female,26.0,S
3,1,1,female,35.0,S
4,0,3,male,35.0,S
...,...,...,...,...,...
886,0,2,male,27.0,S
887,1,1,female,19.0,S
888,0,3,female,NaN,S
889,1,1,male,26.0,C


In [61]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
Embarked      2
dtype: int64

In [62]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

from sklearn.impute import SimpleImputer #step-1
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler #step-2,3
from sklearn.feature_selection import chi2, SelectKBest #setp-4
from sklearn.tree import DecisionTreeClassifier  #step-5

from sklearn.pipeline import Pipeline #pipeline


In [63]:
X = df.iloc[:, 1:]
Y = df.iloc[:, :1]
X_train:pd.DataFrame
X_test:pd.DataFrame
Y_train:pd.Series
Y_test:pd.Series
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.33, random_state=45)
X_train

,Pclass,Sex,Age,Embarked
126,3,male,NaN,Q
369,1,female,24.0,C
61,1,female,38.0,NaN
812,2,male,35.0,S
182,3,male,9.0,S
...,...,...,...,...
725,3,male,20.0,S
607,1,male,27.0,S
544,1,male,50.0,C
643,3,male,NaN,S


In [64]:
t1 = ColumnTransformer([
  ("Age_fill" , SimpleImputer(), [2]),
  ("Embarked_fill" , SimpleImputer(strategy="most_frequent"), [3]) #use index insead of name bcz every ColumnTransformer return np array not ds
], remainder="passthrough") #fill miss values

In [65]:
t2 = ColumnTransformer([
  ("encode", OneHotEncoder(sparse_output=False, handle_unknown="ignore", drop="first"), [1,3])
], remainder="passthrough")

In [66]:
t3 = ColumnTransformer([
  ("MinMaxscale", MinMaxScaler(),slice(0,4))
], remainder="passthrough")

In [67]:
t4 = SelectKBest(score_func=chi2, k=3)

In [68]:
t5 = DecisionTreeClassifier()

In [69]:
#pipline
pipe = Pipeline([
  ("t1", t1),
  ("t2", t2),
  ("t3", t3),
  ("t4", t4),
  ("t5", t5),
])

In [70]:
pipe.fit(X=X_train, y=Y_train) #call fit for model training

,steps,"[('t1', ...), ('t2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Age_fill', ...), ('Embarked_fill', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [71]:
pred = pipe.predict(X_test)

In [72]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(Y_test, pred))
print(classification_report(Y_test, pred))


Accuracy: 0.8508474576271187
              precision    recall  f1-score   support

           0       0.84      0.95      0.89       192
           1       0.87      0.67      0.76       103

    accuracy                           0.85       295
   macro avg       0.86      0.81      0.83       295
weighted avg       0.85      0.85      0.85       295



In [73]:
import pickle
with open("model.pkl", "+wb") as file:
  pickle.dump(pipe, file)